# Baseline Benchmarks: M0 (Beat/Miss) vs M1 (EPS Surprise %)

Two baselines for predicting 5-day post-earnings return direction:

- **M0 — Naive Rule**: Simply predict "Up" if earnings beat consensus, "Down" if miss. No model training — just a rule: `beat → up, miss → down`. This is the simplest possible strategy: look at the earnings headline and guess.
- **M1 — Logistic Regression on Surprise %**: Use the magnitude of EPS surprise as input to a logistic regression. This captures *how much* earnings beat/missed, not just the direction.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 12

conn = sqlite3.connect('../data/hackathon.db')
print('DB connected')

## 1. Build Target Variable

For each of the 91 earnings events, compute the 5-trading-day cumulative return after the earnings date. Label = 1 if positive (stock went up), 0 otherwise.

In [ ]:
earnings = pd.read_sql('SELECT * FROM earnings', conn)
prices = pd.read_sql('SELECT * FROM daily_prices', conn)
prices['date'] = pd.to_datetime(prices['date'])
earnings['earnings_date'] = pd.to_datetime(earnings['earnings_date'])

records = []
for _, evt in earnings.iterrows():
    tk = evt['ticker']
    ed = evt['earnings_date']
    
    tk_prices = prices[prices['ticker'] == tk].sort_values('date')
    
    # First 5 trading days after ED
    post = tk_prices[tk_prices['date'] > ed].head(5)
    if len(post) < 5:
        continue
    
    # Close price on ED (or last trading day before ED) as baseline
    pre = tk_prices[tk_prices['date'] <= ed].tail(1)
    if pre.empty:
        continue
    
    p0 = pre['close'].values[0]
    p5 = post['close'].values[-1]
    ret_5d = (p5 - p0) / p0
    
    records.append({
        'ticker': tk,
        'earnings_date': ed,
        'surprise_pct': evt['surprise_pct'],
        'ret_5d': ret_5d,
        'target': 1 if ret_5d > 0 else 0,
    })

df = pd.DataFrame(records).sort_values('earnings_date').reset_index(drop=True)
print(f'Total events: {len(df)}')
print(f'Target distribution: {df["target"].value_counts().to_dict()}')
print(f'Up ratio: {df["target"].mean():.1%}')
df.head(10)

---

## 2. M0 — Naive Beat/Miss Rule

The simplest possible strategy: if EPS beat consensus (`surprise_pct > 0`), predict stock goes up; if miss, predict down. No model, no training — just a direct rule.

In [ ]:
# M0: beat → predict Up, miss → predict Down
# No model training — just a direct rule

MIN_TRAIN = 40  # same test window for both M0 and M1

X = df[['surprise_pct']].values
y = df['target'].values

m0_pred = (df['surprise_pct'].values > 0).astype(int)

# Evaluate on test set (events 41-91, same window M1 will use)
m0_pred_test = m0_pred[MIN_TRAIN:]
m0_true_test = y[MIN_TRAIN:]

m0_acc = accuracy_score(m0_true_test, m0_pred_test)
m0_f1 = f1_score(m0_true_test, m0_pred_test)
m0_auc = roc_auc_score(m0_true_test, m0_pred_test.astype(float))

print('=' * 50)
print('M0 Naive Rule: Beat → Up, Miss → Down')
print('=' * 50)
print(f'Accuracy:  {m0_acc:.3f}')
print(f'F1 Score:  {m0_f1:.3f}')
print(f'AUC-ROC:   {m0_auc:.3f}')
print()
print('Classification Report:')
print(classification_report(m0_true_test, m0_pred_test, target_names=['Down', 'Up'], zero_division=0))
print('Confusion Matrix:')
cm = confusion_matrix(m0_true_test, m0_pred_test)
print(cm)

# Quick interpretation
beat_count = (m0_pred_test == 1).sum()
miss_count = (m0_pred_test == 0).sum()
print(f'\nOf {len(m0_pred_test)} events: {beat_count} beats, {miss_count} misses')
print(f'When beat: {cm[1,1]}/{beat_count} actually went up ({cm[1,1]/beat_count:.1%})')
if miss_count > 0:
    print(f'When miss: {cm[0,0]}/{miss_count} actually went down ({cm[0,0]/miss_count:.1%})')

---

## 3. M1 — Logistic Regression on EPS Surprise %

Unlike M0's binary rule, M1 uses the *magnitude* of surprise via logistic regression with walk-forward validation.

In [ ]:
# M1: Walk-forward logistic regression on surprise_pct
# X, y, MIN_TRAIN already defined in M0 cell

y_true_list = []
y_pred_list = []
y_prob_list = []

for t in range(MIN_TRAIN, len(df)):
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X[:t], y[:t])
    
    y_pred_list.append(model.predict(X[t:t+1])[0])
    y_prob_list.append(model.predict_proba(X[t:t+1])[0, 1])
    y_true_list.append(y[t])

y_true = np.array(y_true_list)
y_pred = np.array(y_pred_list)
y_prob = np.array(y_prob_list)

m1_acc = accuracy_score(y_true, y_pred)
m1_f1 = f1_score(y_true, y_pred)
m1_auc = roc_auc_score(y_true, y_prob)

print('=' * 50)
print('M1 Logistic Regression (surprise_pct)')
print('=' * 50)
print(f'Accuracy:  {m1_acc:.3f}')
print(f'F1 Score:  {m1_f1:.3f}')
print(f'AUC-ROC:   {m1_auc:.3f}')
print()
print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=['Down', 'Up'], zero_division=0))
print('Confusion Matrix:')
print(confusion_matrix(y_true, y_pred))

## 4. M1 Model Coefficients & Interpretation

In [ ]:
# Train a final model on ALL data to inspect coefficients
final_model = LogisticRegression(random_state=42, max_iter=1000)
final_model.fit(X, y)

coef = final_model.coef_[0][0]
intercept = final_model.intercept_[0]

print('Logistic Regression: P(Up) = sigmoid(b0 + b1 * surprise_pct)')
print(f'  Intercept (b0): {intercept:.4f}')
print(f'  Coefficient (b1): {coef:.4f}')
print()

# Interpretation
print('Interpretation:')
print(f'  - Intercept = {intercept:.4f} → at surprise_pct=0 (meets consensus exactly),')
base_prob = 1 / (1 + np.exp(-intercept))
print(f'    the predicted P(Up) = {base_prob:.1%}')
print(f'  - Coefficient = {coef:.4f} → each +1% EPS surprise increases the log-odds by {coef:.4f}')

odds_ratio = np.exp(coef)
print(f'  - Odds ratio = exp({coef:.4f}) = {odds_ratio:.4f}')
print(f'    → each +1% surprise multiplies the odds of going up by {odds_ratio:.2f}x')
print()

# Show predicted probability at key surprise levels
print('Predicted P(Up) at key surprise levels:')
for sp in [-10, -5, 0, 5, 10, 20, 50]:
    prob = 1 / (1 + np.exp(-(intercept + coef * sp)))
    print(f'  surprise_pct = {sp:+4d}%  →  P(Up) = {prob:.1%}')

## 5. M0 vs M1 Comparison Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Baseline Benchmarks: M0 (Beat/Miss Rule) vs M1 (Logistic Regression)', fontsize=15, fontweight='bold')

# ── 1. ROC Curve: M0 vs M1 ──
ax = axes[0, 0]
fpr1, tpr1, _ = roc_curve(y_true, y_prob)
ax.plot(fpr1, tpr1, 'b-', linewidth=2, label=f'M1 LogReg (AUC={m1_auc:.3f})')
# M0 is a single point on ROC space (no threshold to vary)
m0_fpr = 1 - (cm[0,0] / (cm[0,0] + cm[0,1]))  # FPR
m0_tpr = cm[1,1] / (cm[1,0] + cm[1,1])  # TPR
ax.plot(m0_fpr, m0_tpr, 'rs', markersize=12, label=f'M0 Naive Rule (AUC={m0_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)

# ── 2. Metrics Comparison Bar Chart: M0 vs M1 ──
ax = axes[0, 1]
x_pos = np.arange(3)
width = 0.3
bars_m0 = ax.bar(x_pos - width/2, [m0_acc, m0_f1, m0_auc], width, label='M0 Naive', color='#F44336', alpha=0.8)
bars_m1 = ax.bar(x_pos + width/2, [m1_acc, m1_f1, m1_auc], width, label='M1 LogReg', color='#2196F3', alpha=0.8)
for bars in [bars_m0, bars_m1]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', fontsize=10, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(['Accuracy', 'F1', 'AUC'])
ax.set_ylim(0, 1)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4, label='Random')
ax.set_title('M0 vs M1 Metrics Comparison')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# ── 3. Surprise vs Return Scatter ──
ax = axes[1, 0]
test_df = df.iloc[MIN_TRAIN:].copy()
test_df['m0_correct'] = (m0_pred_test == m0_true_test)
colors = test_df['target'].map({1: '#4CAF50', 0: '#F44336'})
ax.scatter(test_df['surprise_pct'], test_df['ret_5d'] * 100,
           c=colors, alpha=0.6, s=50, edgecolors='white', linewidth=0.5)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('EPS Surprise %')
ax.set_ylabel('5-Day Return %')
ax.set_title('EPS Surprise vs Return (green=Up, red=Down)')
ax.text(0.95, 0.95, 'Beat & Up', transform=ax.transAxes, ha='right', va='top', fontsize=9, color='green', alpha=0.7)
ax.text(0.05, 0.95, 'Miss & Up', transform=ax.transAxes, ha='left', va='top', fontsize=9, color='orange', alpha=0.7)
ax.text(0.95, 0.05, 'Beat & Down', transform=ax.transAxes, ha='right', va='bottom', fontsize=9, color='orange', alpha=0.7)
ax.text(0.05, 0.05, 'Miss & Down', transform=ax.transAxes, ha='left', va='bottom', fontsize=9, color='green', alpha=0.7)
ax.grid(alpha=0.3)

# ── 4. Cumulative Trading Returns: M0 vs M1 with Gross Profit ──
ax = axes[1, 1]
test_df = test_df.copy()
test_df['m0_strat_ret'] = np.where(m0_pred_test == 1, test_df['ret_5d'], -test_df['ret_5d'])
test_df['m1_strat_ret'] = np.where(y_pred == 1, test_df['ret_5d'], -test_df['ret_5d'])
test_df['cum_m0'] = (1 + test_df['m0_strat_ret']).cumprod()
test_df['cum_m1'] = (1 + test_df['m1_strat_ret']).cumprod()
test_df['cum_buyhold'] = (1 + test_df['ret_5d']).cumprod()

m0_gross = (test_df['cum_m0'].iloc[-1] - 1) * 100
m1_gross = (test_df['cum_m1'].iloc[-1] - 1) * 100
bh_gross = (test_df['cum_buyhold'].iloc[-1] - 1) * 100

x_range = range(len(test_df))
ax.plot(x_range, test_df['cum_m0'].values, 'r-', linewidth=2,
        label=f'M0 Naive ({m0_gross:+.1f}%)')
ax.plot(x_range, test_df['cum_m1'].values, 'b-', linewidth=2,
        label=f'M1 LogReg ({m1_gross:+.1f}%)')
ax.plot(x_range, test_df['cum_buyhold'].values, 'gray', linewidth=1, alpha=0.5,
        label=f'Buy & Hold ({bh_gross:+.1f}%)')
ax.axhline(y=1.0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Event #')
ax.set_ylabel('Cumulative Return')
ax.set_title('Simulated Trading Returns')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/m0_m1_baseline_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/m0_m1_baseline_benchmark.png')

## 6. Summary

| Metric | M0 (Beat/Miss Rule) | M1 (LogReg on Surprise %) | Random |
|--------|---------------------|---------------------------|--------|
| Accuracy | — | — | 0.500 |
| F1 | — | — | — |
| AUC-ROC | — | — | 0.500 |

*(Values filled in after running the notebook)*

**Key takeaways:**

1. **M0 is the "gut feeling" baseline** — it answers: *"If you just looked at the earnings headline (beat or miss) and immediately guessed the stock direction, how often would you be right?"* This is the bar that any smarter model must clear.

2. **M1 adds magnitude** — knowing the company beat by 2% vs 50% should matter. The logistic regression captures this continuous relationship, but with only 1 feature and 91 samples, the improvement over M0 may be modest.

3. **Both baselines use only earnings data** — no sentiment, no cross-company information. This sets the stage for M2 (+sentiment) and M3 (+spillover network) to demonstrate whether text signals add value beyond what the market already prices in from the earnings number itself.